# visualize the features of Vietnam, MIMIC-III, and Med-MEC

In [3]:
import os
import glob
import pandas as pd
import numpy as np
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

# --- Expected PPG feature list ---
ppg_feature_names = [
    'Tpi','Tpp','Tsys','Tdia','Tsp','Tdp','deltaT',
    'Tsw10','Tsw25','Tsw33','Tsw50','Tsw66','Tsw75','Tsw90',
    'Tdw10','Tdw25','Tdw33','Tdw50','Tdw66','Tdw75','Tdw90',
    'Tpw10','Tpw25','Tpw33','Tpw50','Tpw66','Tpw75','Tpw90',
    'Asp','Adn','Adp','Aoff','AUCpi','AUCsys','AUCdia','SQI',
    'HRV_MeanNN','HRV_SDNN','HRV_SDANN1','HRV_SDNNI1','HRV_SDANN2','HRV_SDNNI2',
    'HRV_SDANN5','HRV_SDNNI5','HRV_RMSSD','HRV_SDSD','HRV_CVNN','HRV_CVSD',
    'HRV_MedianNN','HRV_MadNN','HRV_MCVNN','HRV_IQRNN','HRV_SDRMSSD',
    'HRV_Prc20NN','HRV_Prc80NN','HRV_pNN50','HRV_pNN20','HRV_MinNN','HRV_MaxNN',
    'HRV_HTI','HRV_TINN','HRV_ULF','HRV_VLF','HRV_LF','HRV_HF','HRV_VHF',
    'HRV_TP','HRV_LFHF','HRV_LFn','HRV_HFn','HRV_LnHF',
    'HRV_SD1','HRV_SD2','HRV_SD1SD2','HRV_S',
    'HRV_CSI','HRV_CVI','HRV_CSI_Modified','HRV_PIP','HRV_IALS',
    'HRV_PSS','HRV_PAS','HRV_GI','HRV_SI','HRV_AI','HRV_PI',
    'HRV_C1d','HRV_C1a','HRV_SD1d','HRV_SD1a','HRV_C2d','HRV_C2a',
    'HRV_SD2d','HRV_SD2a','HRV_Cd','HRV_Ca','HRV_SDNNd','HRV_SDNNa',
    'HRV_DFA_alpha1',
    'HRV_MFDFA_alpha1_Width','HRV_MFDFA_alpha1_Peak','HRV_MFDFA_alpha1_Mean',
    'HRV_MFDFA_alpha1_Max','HRV_MFDFA_alpha1_Delta','HRV_MFDFA_alpha1_Asymmetry',
    'HRV_MFDFA_alpha1_Fluctuation','HRV_MFDFA_alpha1_Increment',
    'HRV_ApEn','HRV_SampEn','HRV_ShanEn','HRV_FuzzyEn',
    'HRV_MSEn','HRV_CMSEn','HRV_RCMSEn',
    'HRV_CD','HRV_HFD','HRV_KFD','HRV_LZC',
    # Possibly also 'HRV_DFA_alpha2' if present
]


In [4]:
import numpy as np
from sklearn.manifold import TSNE
import sklearn
print("sklearn version:", sklearn.__version__)
print(TSNE)

sklearn version: 1.7.2
<class 'sklearn.manifold._t_sne.TSNE'>


# Helper Function to Load FIRST ROW of Each CSV

In [6]:
from tqdm.notebook import tqdm

def load_first_row_features(pattern, site_label):
    files = sorted(glob.glob(pattern))[:160]
    vectors = []
    labels = []

    for fp in tqdm(files):
        try:
            df = pd.read_csv(fp)
        except Exception as e:
            print(f"Skipping {fp} — read error: {e}")
            continue

        # extract first row using only expected features
        row = []
        for col in ppg_feature_names:
            if col in df.columns:
                row.append(float(df[col].iloc[0]))
            else:
                row.append(0.0)   # missing → fill with zero

        vectors.append(row)
        labels.append(site_label)

    return np.array(vectors), np.array(labels)


# Load Vietnam + MIMIC

In [7]:
vn_X, vn_y = load_first_row_features("../../data/PPG/*.csv", site_label="Vietnam")
mi_X, mi_y = load_first_row_features("../../code_xg/preprocess_MIMIC-III/ppg_features_by_icu_v2/*.csv", site_label="MIMIC")

X = np.vstack([vn_X, mi_X])
y = np.concatenate([vn_y, mi_y])

print("Total samples:", X.shape)
print("Vietnam:", vn_X.shape, "MIMIC:", mi_X.shape)



  0%|          | 0/160 [00:00<?, ?it/s]

  0%|          | 0/160 [00:00<?, ?it/s]

Total samples: (320, 118)
Vietnam: (160, 118) MIMIC: (160, 118)


In [16]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Before cleaning: NaNs =", np.isnan(X).sum(), "infs =", np.isinf(X).sum())


Before cleaning: NaNs = 0 infs = 0


/home/engs2588/miniconda3/envs/vital/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/home/engs2588/miniconda3/envs/vital/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/home/engs2588/miniconda3/envs/vital/lib/python3.10/site-packages/sklearn/utils/extmath.py:1163: RuntimeWarning: overflow encountered in square
  temp **= 2
/home/engs2588/miniconda3/envs/vital/lib/python3.10/site-packages/sklearn/utils/extmath.py:1169: RuntimeWarning: overflow encountered in square
  new_unnormalized_variance -= correction**2 / new_sample_count
/home/engs2588/miniconda3/envs/vital/lib/python3.10/site-packages/sklearn/utils/extmath.py:1169: RuntimeWarning: invalid value encountered in subtract
  new_unnormalized_variance -= correction**2 / new_sample_count
/home/eng

In [13]:
## Run t-SNE
X = np.nan_to_num(X, nan=0.0)
# (D) t-SNE on PCA output
tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate='auto',
    # init='pca',
    random_state=42,
    verbose=1,
)

X_emb = tsne.fit_transform(X)

[t-SNE] Computing 91 nearest neighbors...
[t-SNE] Indexed 320 samples in 0.000s...
[t-SNE] Computed neighbors for 320 samples in 0.004s...
[t-SNE] Computed conditional probabilities for sample 320 / 320
[t-SNE] Mean sigma: 0.000000


/home/engs2588/miniconda3/envs/vital/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/home/engs2588/miniconda3/envs/vital/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/home/engs2588/miniconda3/envs/vital/lib/python3.10/site-packages/sklearn/manifold/_t_sne.py:104: RuntimeWarning: overflow encountered in cast
  distances_data = distances_data.astype(np.float32, copy=False)
/home/engs2588/miniconda3/envs/vital/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/home/engs2588/miniconda3/envs/vital/lib/python3.10/site-packages/numpy/_core/_methods.py:135: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(arr, axis,

ValueError: array must not contain infs or NaNs

In [2]:
plt.figure(figsize=(8,6))

plt.scatter(
    X_emb[y=="Vietnam", 0],
    X_emb[y=="Vietnam", 1],
    c="blue", label="Vietnam", alpha=0.6
)

plt.scatter(
    X_emb[y=="MIMIC", 0],
    X_emb[y=="MIMIC", 1],
    c="red", label="MIMIC", alpha=0.6
)

plt.legend()
plt.title("t-SNE of PPG First-Row Feature Vectors (Vietnam vs MIMIC)")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.tight_layout()
plt.savefig("tsne_ppg_vietnam_vs_mimic.png", dpi=300)

plt.show()


NameError: name 'plt' is not defined